# Week 07 — Home exercise 5: Cleaning decisions

**Solution proposal.**

Four cleaning questions, none of which has a single correct answer. What is being demonstrated is choosing deliberately and writing down why.

In [1]:
import pandas as pd

co2 = pd.read_csv("../data/co2_emissions.csv")

## 1. Where is it missing? 2005 against 2023

In [2]:
print(f"{'column':<16}{'2005':>8}{'2023':>8}")

for column in ["co2_total", "gdp_pc", "renew_energy"]:
    n_2005 = co2[co2["year"] == 2005][column].isna().sum()
    n_2023 = co2[co2["year"] == 2023][column].isna().sum()
    print(f"{column:<16}{n_2005:>8}{n_2023:>8}")

column              2005    2023
co2_total             14      14
gdp_pc                 8      13
renew_energy           7     260


`co2_total` is missing 14 times in both years — the same handful of entities, year in and year
out. That is a **coverage gap**: some places are simply not in this series.

`renew_energy` goes from 7 to 260. That is not a coverage gap, it is a **publication lag**: the series
does not exist yet for the most recent years.

The two look identical in a total count of missing values and mean completely different things.
Counting is not enough; you have to ask where.

## 2. What each `dropna` costs

In [3]:
print(f"{'all rows':<36}{len(co2):>6}")
print(f"{'dropna()':<36}{len(co2.dropna()):>6}")
print(f"{'dropna(subset=[co2_total])':<36}{len(co2.dropna(subset=['co2_total'])):>6}")
print(f"{'dropna(subset=[co2_total, gdp_pc])':<36}{len(co2.dropna(subset=['co2_total', 'gdp_pc'])):>6}")

all rows                              6240
dropna()                              5150
dropna(subset=[co2_total])            5904
dropna(subset=[co2_total, gdp_pc])    5792


For a study of emissions against income, **the third one is right**: it keeps every row where
both variables I need are present, and 5 792 of 6 240 is a good deal.

`dropna()` is wrong because it throws away 1 090 rows for holes in columns I am not using — and since
`renew_energy` is missing for all of 2023, it deletes the most recent year of the study entirely
without mentioning it.

`dropna(subset=["co2_total"])` is wrong in the other direction: it leaves 112 rows with no GDP figure,
which will then vanish silently from any calculation that uses both. Better to remove them on purpose
than to have them disappear later.

## 3. Filling with the mean, and what it costs

In [4]:
before = co2["renew_energy"]
after = co2["renew_energy"].fillna(co2["renew_energy"].mean())

print(f"{'':<8}{'mean':>10}{'std':>10}{'count':>8}")
print(f"{'before':<8}{before.mean():>10.2f}{before.std():>10.2f}{before.count():>8}")
print(f"{'after':<8}{after.mean():>10.2f}{after.std():>10.2f}{after.count():>8}")

              mean       std   count
before       30.28     28.81    5605
after        30.28     27.30    6240


The mean is unchanged, which is the whole appeal of filling with the mean — and it is a trap.
The standard deviation fell from 28.81 to 27.30, because 635 values that used to be unknown now sit
exactly at the average, adding nothing to the spread.

Why that matters: almost every statistic you would go on to compute — a correlation, a regression
slope, a confidence interval — depends on the spread. Filling holes with the mean makes the data look
more certain than it is, and it does so invisibly, because the mean you would check has not moved.

Here it would be worse still. The missing values are concentrated in 2021–2023, so filling them with a
24-year average invents a flat trend in exactly the years anyone would want to look at.

## 4. Text that should be numbers

In [5]:
messy = pd.Series(["41.6", "434.3", "n/a", "", "43.5", "1 200", "12.4"])
converted = pd.to_numeric(messy, errors="coerce")

for raw, value in zip(messy, converted):
    print(f"{repr(raw):<10} -> {value}")

print()
print("became NaN:", converted.isna().sum(), "of", len(messy))

'41.6'     -> 41.6
'434.3'    -> 434.3
'n/a'      -> nan
''         -> nan
'43.5'     -> 43.5
'1 200'    -> nan
'12.4'     -> 12.4

became NaN: 3 of 7


Three failed. Two of them should have: `"n/a"` and the empty string are genuinely not numbers.

The third is the interesting one. `"1 200"` is a number written with a space as a thousands
separator, which is normal in Norwegian and in a great deal of exported data. `errors="coerce"` turned
it into `NaN`, so a perfectly good observation was destroyed silently by a cleaning step meant to be
safe.

That is the cost of `coerce`: it cannot tell "this is not a number" from "this is a number written in
a way I do not recognize". Look at what became `NaN` before accepting the result — here the fix is
`messy.str.replace(" ", "")` before converting.

## The thread running through all four

Every question here had a defensible answer that was also wrong, and in every case the wrong answer
was the one that produced no error:

1. counting missing values without asking where they are
2. `dropna()` with no subset, which quietly deletes a whole year
3. filling with the mean, which leaves the mean unchanged and the spread wrong
4. `errors="coerce"`, which turned a valid number into a hole

Cleaning is the part of an analysis where mistakes are cheapest to make and most expensive to find,
because nothing crashes and every number that comes out still looks like a number. The defense is not
to be more careful; it is to check the shape and the summary statistics before and after every step,
and to write down which decision you made and why.

### What this notebook does NOT do

- It does not fix anything. Every section reports and explains; none of it is assembled into a cleaned
  dataset, which is exercise 6's job.
- The 14 entities with no emissions data are never identified by name. They should be: a coverage gap
  that always affects the same places is a bias, and knowing which places are missing tells you which
  conclusions you cannot draw.
- It treats every missing value as equally missing. In practice "not collected", "collected but not
  published yet" and "genuinely zero" are different things that all arrive as `NaN`, and no amount of
  pandas will tell them apart. Only the documentation for the dataset can.